In [70]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,f1_score,precision_score,recall_score,roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
import pandas as pd

In [71]:
df=pd.read_csv('../data/processed/match_data.csv')
df.head()


,match_id,batting_team,bowling_team,toss_winner,toss_decision,venue,city,season,match_won_by,player_of_match,batting_team_home,bowling_team_home,batting_team_won_toss,is_knockout,stage_clean,year,month,day
0,335982,Kolkata Knight Riders,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,M Chinnaswamy Stadium,Bangalore,2007/08,Kolkata Knight Riders,BB McCullum,0,1,0,0,League,2008,4,18
1,335983,Chennai Super Kings,Punjab Kings,Chennai Super Kings,bat,Punjab Cricket Association Stadium,Chandigarh,2007/08,Chennai Super Kings,MEK Hussey,0,0,1,0,League,2008,4,19
2,335984,Rajasthan Royals,Delhi Capitals,Rajasthan Royals,bat,Feroz Shah Kotla,Delhi,2007/08,Delhi Capitals,MF Maharoof,0,1,1,0,League,2008,4,19
3,335985,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,Wankhede Stadium,Mumbai,2007/08,Royal Challengers Bengaluru,MV Boucher,1,0,1,0,League,2008,4,20
4,335987,Punjab Kings,Rajasthan Royals,Punjab Kings,bat,Sawai Mansingh Stadium,Jaipur,2007/08,Rajasthan Royals,SR Watson,0,1,1,0,League,2008,4,21


In [72]:
X=df.drop(['match_won_by','player_of_match','match_id'], axis=1)
y=df['match_won_by']
y.value_counts()

match_won_by
Mumbai Indians                 138
Chennai Super Kings            133
Kolkata Knight Riders          120
Royal Challengers Bengaluru    119
Punjab Kings                   108
Rajasthan Royals               106
Delhi Capitals                 104
Sunrisers Hyderabad             87
Gujarat Titans                  39
Lucknow Super Giants            32
Name: count, dtype: int64

In [73]:
X.nunique()

batting_team             10
bowling_team             10
toss_winner              10
toss_decision             2
venue                    37
city                     34
season                   19
batting_team_home         2
bowling_team_home         2
batting_team_won_toss     2
is_knockout               2
stage_clean               3
year                     19
month                     7
day                      31
dtype: int64

In [74]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 986 entries, 0 to 985
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   batting_team           986 non-null    object
 1   bowling_team           986 non-null    object
 2   toss_winner            986 non-null    object
 3   toss_decision          986 non-null    object
 4   venue                  986 non-null    object
 5   city                   986 non-null    object
 6   season                 986 non-null    object
 7   batting_team_home      986 non-null    int64 
 8   bowling_team_home      986 non-null    int64 
 9   batting_team_won_toss  986 non-null    int64 
 10  is_knockout            986 non-null    int64 
 11  stage_clean            986 non-null    object
 12  year                   986 non-null    int64 
 13  month                  986 non-null    int64 
 14  day                    986 non-null    int64 
dtypes: int64(7), object(8)


In [75]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df["match_won_by"])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

In [76]:
cols_one_hot = ['batting_team','bowling_team','toss_winner','toss_decision','venue','city','season','stage_clean']
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cols_one_hot)
    ],
    remainder='passthrough'
)
X_train_processed=preprocessor.fit_transform(X_train)
X_test_processed=preprocessor.transform(X_test)

In [77]:
models={
    "logisticregression":LogisticRegression(C=1.0,
    penalty='l2',
    solver='lbfgs',
    max_iter=1000,
    random_state=42
    ),
    "randomforest":RandomForestClassifier(
      n_estimators=300,
      max_depth=10,
      min_samples_split=10,
      min_samples_leaf=5,
      max_features='sqrt',
      random_state=42
    ),
    "xgboost":XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=1,
    random_state=42
      )
}


In [78]:
results = []

best_model = None
best_name = ""
best_auc = 0

for name, model in models.items():

    # Train
    model.fit(X_train_processed, y_train)

    # Predictions
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    roc_auc = roc_auc_score(
        y_test,
        y_prob,
        multi_class="ovr",
        average="weighted"
    )

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ])

    print("=" * 60)
    print(name)
    print("=" * 60)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")

    print("\nClassification Report")
    print(classification_report(
        y_test,
        y_pred,
        target_names=le.classes_
    ))

    print("\nConfusion Matrix")
    print(confusion_matrix(y_test, y_pred))

    print()

    # Save best model
    if roc_auc > best_auc:
        best_auc = roc_auc
        best_model = model
        best_name = name

c:\Users\SRIHARSHITH\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


logisticregression
Accuracy : 0.5000
Precision: 0.4957
Recall   : 0.5000
F1 Score : 0.4875
ROC-AUC  : 0.9310

Classification Report
                             precision    recall  f1-score   support

        Chennai Super Kings       0.57      0.52      0.55        23
             Delhi Capitals       0.39      0.43      0.41        21
             Gujarat Titans       0.50      0.43      0.46         7
      Kolkata Knight Riders       0.54      0.66      0.59        29
       Lucknow Super Giants       0.50      0.09      0.15        11
             Mumbai Indians       0.52      0.64      0.57        22
               Punjab Kings       0.37      0.30      0.33        23
           Rajasthan Royals       0.61      0.73      0.67        15
Royal Challengers Bengaluru       0.60      0.60      0.60        30
        Sunrisers Hyderabad       0.29      0.29      0.29        17

                   accuracy                           0.50       198
                  macro avg       0.49

c:\Users\SRIHARSHITH\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\SRIHARSHITH\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\SRIHARSHITH\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\SRIHARSHITH\a

randomforest
Accuracy : 0.5303
Precision: 0.5072
Recall   : 0.5303
F1 Score : 0.5081
ROC-AUC  : 0.9374

Classification Report
                             precision    recall  f1-score   support

        Chennai Super Kings       0.62      0.78      0.69        23
             Delhi Capitals       0.42      0.52      0.47        21
             Gujarat Titans       0.50      0.29      0.36         7
      Kolkata Knight Riders       0.59      0.66      0.62        29
       Lucknow Super Giants       0.00      0.00      0.00        11
             Mumbai Indians       0.47      0.73      0.57        22
               Punjab Kings       0.54      0.30      0.39        23
           Rajasthan Royals       0.47      0.53      0.50        15
Royal Challengers Bengaluru       0.64      0.53      0.58        30
        Sunrisers Hyderabad       0.44      0.47      0.46        17

                   accuracy                           0.53       198
                  macro avg       0.47      

In [ ]:
import joblib
joblib.dump(best_model,"winner_model.pkl")
joblib.dump(preprocessor,"winner_preprocessor.pkl")
joblib.dump(le,"winner_label-encoder.pkl")
print(f"\nBest Model : {best_name}")
print(f"ROC-AUC    : {best_auc:.4f}")
print("Model Saved Successfully")
results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ]
)

results_df = results_df.sort_values(
    by="ROC_AUC",
    ascending=False
)

results_df


Best Model : randomforest
ROC-AUC    : 0.9374
accuracy 0.45454545454545453
Model Saved Successfully


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
1,randomforest,0.530303,0.507228,0.530303,0.508110,0.937435
2,xgboost,0.454545,0.461115,0.454545,0.447511,0.931202
0,logisticregression,0.500000,0.495711,0.500000,0.487456,0.931034


In [102]:
new_match = pd.DataFrame({
    'batting_team': ['Chennai Super Kings'],
    'bowling_team': ['Mumbai Indians'],
    'toss_winner': ['Chennai Super Kings'],
    'toss_decision': ['bat'],
    'venue': ['Wankhede'],
    'city': ['Mumbai'],
    'season': ['2026'],
    'stage_clean': ['group stage'],
    'year': [2026],
    'month': [4],
    'day':[19],
    'batting_team_home': [0],
    'bowling_team_home': [0],
    'batting_team_won_toss': [0],
    'is_knockout': [1]
})

In [103]:
new_match_encoded = preprocessor.transform(new_match)
prediction = model.predict(new_match_encoded)
winner =le.inverse_transform(prediction)
print("Predicted Winner:", winner[0])

Predicted Winner: Chennai Super Kings
